# 📊 Chapter 9: Common Distributions Part 2
*Tier 1 — Foundations | All Tracks*

> **🎬 Hook:** How many customers walk into a store in an hour? How long until the next bus arrives?
> There are distributions *designed* for these exact questions.

**🎯 Objectives:** Understand and apply Poisson, Uniform, and Exponential distributions.

## 📖 Section 1 — Concept Review

### Poisson(λ) — For COUNTS of rare events in a fixed interval
$$P(X=k) = \frac{e^{-\lambda} \lambda^k}{k!} \quad k = 0,1,2,\ldots$$
- E[X] = λ, Var(X) = λ (mean = variance!)
- Use when: counting arrivals, defects, events per time unit

### Uniform(a,b) — All values equally likely in [a,b]
$$f(x) = \frac{1}{b-a} \quad a \leq x \leq b$$
- E[X] = (a+b)/2, Var(X) = (b-a)²/12

### Exponential(λ) — Waiting TIME between Poisson events
$$f(x) = \lambda e^{-\lambda x} \quad x \geq 0$$
- E[X] = 1/λ, Var(X) = 1/λ²
- **Memoryless:** P(X > s+t | X > s) = P(X > t)
- If events arrive at rate λ, waiting time ~ Exp(λ)

### The Poisson-Exponential Connection
If customers arrive at rate λ per hour (Poisson), then the time BETWEEN arrivals ~ Exponential(λ).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid")
np.random.seed(42)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# ── Poisson: different λ ──
k = np.arange(0, 25)
for ax, (lam, color) in zip([axes[0,0], axes[0,1]],
    [(2, '#3498db'), (8, '#e74c3c')]):
    ax.bar(k, stats.poisson.pmf(k, lam), color=color, edgecolor='white', lw=1.5)
    ax.axvline(lam, color='black', linestyle='--', lw=2, label=f'E[X]=λ={lam}')
    ax.set_title(f'Poisson(λ={lam})', fontweight='bold')
    ax.set_xlabel('k'); ax.set_ylabel('P(X=k)')
    ax.legend()

# ── Uniform ──
x = np.linspace(-0.5, 5.5, 300)
for ax, (a, b, color) in zip([axes[0,2]],
    [(0, 5, '#27ae60')]):
    ax.plot(x, stats.uniform.pdf(x, a, b-a), color=color, lw=3)
    ax.fill_between(x, stats.uniform.pdf(x, a, b-a), alpha=0.3, color=color)
    ax.axvline((a+b)/2, color='red', linestyle='--', lw=2, label=f'E[X]={(a+b)/2}')
    ax.set_title(f'Uniform({a},{b})', fontweight='bold')
    ax.set_xlabel('x'); ax.set_ylabel('f(x)')
    ax.legend()
    ax.set_ylim(0, 0.5)

# ── Exponential ──
x_exp = np.linspace(0, 10, 300)
for ax, (lam, color) in zip([axes[1,0], axes[1,1]],
    [(0.5, '#9b59b6'), (2, '#e67e22')]):
    ax.plot(x_exp, stats.expon.pdf(x_exp, scale=1/lam), color=color, lw=3)
    ax.fill_between(x_exp, stats.expon.pdf(x_exp, scale=1/lam), alpha=0.3, color=color)
    ax.axvline(1/lam, color='black', linestyle='--', lw=2, label=f'E[X]=1/λ={1/lam:.1f}')
    ax.set_title(f'Exponential(λ={lam})', fontweight='bold')
    ax.set_xlabel('x (time)'); ax.set_ylabel('f(x)')
    ax.legend()

# ── All three together: comparison ──
x_all = np.linspace(0, 15, 300)
axes[1,2].plot(x_all, stats.norm.pdf(x_all, 7, 2),    label='Normal(7,2)', lw=2.5)
axes[1,2].plot(x_all, stats.expon.pdf(x_all, scale=3), label='Exponential(λ=1/3)', lw=2.5)
axes[1,2].plot(x_all, stats.uniform.pdf(x_all, 0, 14), label='Uniform(0,14)', lw=2.5)
axes[1,2].set_title('Shape Comparison: All with E[X]≈7', fontweight='bold')
axes[1,2].set_xlabel('x'); axes[1,2].set_ylabel('f(x)')
axes[1,2].legend(fontsize=8)

plt.suptitle("Poisson, Uniform, Exponential Distributions", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('ch09_distributions_p2.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Poisson-Exponential connection: bus arrivals
np.random.seed(42)
lambda_rate = 3  # buses per hour

# Simulate 1 hour: count arrivals (Poisson)
# Then simulate inter-arrival times (Exponential)
n_hours = 10_000

# Count method (Poisson)
counts = np.random.poisson(lambda_rate, n_hours)

# Time method (Exponential inter-arrivals)
inter_arrival_times = np.random.exponential(scale=1/lambda_rate, size=100_000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

k = np.arange(0, 12)
ax1.bar(k, [(counts == ki).mean() for ki in k], color='#3498db', edgecolor='white', label='Simulated')
ax1.plot(k, stats.poisson.pmf(k, lambda_rate), 'ro-', markersize=6, lw=2, label=f'Poisson(λ={lambda_rate})')
ax1.set_title('🚌 Buses per Hour: Poisson', fontweight='bold')
ax1.set_xlabel('Buses per hour'); ax1.set_ylabel('Probability')
ax1.legend()

x = np.linspace(0, 3, 200)
ax2.hist(inter_arrival_times, bins=50, density=True, alpha=0.5, color='#e74c3c', label='Simulated')
ax2.plot(x, stats.expon.pdf(x, scale=1/lambda_rate), 'b-', lw=3,
         label=f'Exponential(λ={lambda_rate})')
ax2.axvline(1/lambda_rate, color='black', linestyle='--', lw=2,
            label=f'E[wait] = {1/lambda_rate:.2f} hr = 20 min')
ax2.set_title('🚌 Time Between Buses: Exponential', fontweight='bold')
ax2.set_xlabel('Hours between buses'); ax2.set_ylabel('Density')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('ch09_poisson_exp_connection.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Avg buses/hour: {counts.mean():.2f} (theory: {lambda_rate})")
print(f"Avg inter-arrival: {inter_arrival_times.mean():.3f} hr (theory: {1/lambda_rate:.3f} hr)")

## ✏️ Section 6 — Practice Problems

**P1:** Calls arrive at a call center at 10/hour. P(exactly 8 calls in an hour)?
**P2:** A server processes requests in U(0.5, 2.5) seconds. E[time]? P(time < 1 sec)?
**P3:** A machine fails on average every 200 hours (Exponential). P(survives 100 hours)?

<details><summary>💡 Solutions</summary>

**P1:** Poisson(10): P(X=8) = e⁻¹⁰·10⁸/8! ≈ **0.113**

**P2:** E = (0.5+2.5)/2 = **1.5 sec**, P(X<1) = (1-0.5)/(2.5-0.5) = **0.25**

**P3:** Exponential(λ=1/200): P(X>100) = e^(-100/200) = e^(-0.5) ≈ **0.607**
</details>

In [ ]:
# Solutions
print("P1:", stats.poisson.pmf(8, 10))
print("P2 E[X]:", (0.5+2.5)/2, "  P(X<1):", stats.uniform.cdf(1, 0.5, 2.0))
print("P3:", stats.expon.sf(100, scale=200))

## 🎯 Recap
1. **Poisson**: counts of events in fixed interval; mean = variance = λ.
2. **Uniform**: all values equally likely; maximum entropy for a bounded range.
3. **Exponential**: waiting times; memoryless; connected to Poisson arrivals.

**Next:** [Chapter 10 — The Normal Distribution]